In [1]:
import json
import os
import pandas as pd
import xlwings as xw

from datetime import date
from websocket import create_connection

from Class_xlWings import *
xlw  = xlWings()

In [2]:
deribit_url = 'wss://www.deribit.com/ws/api/v2'
ws = create_connection(deribit_url)

In [3]:
def file_maintenance(xlApp, save_date):
    core_path = r"C:\Users\micha\Market_Data_Pulls"
    core_filename = "Daily_Deribit_Option_Snapshot_BTC_"
    filename_ext = ".xlsx"

    open_filename = core_filename + "Template" + filename_ext
    open_address = os.path.join(core_path, open_filename)

    wb = xlApp.books.open(open_address)

    save_filename = core_filename + save_date.strftime('%Y%m%d') + filename_ext
    save_address = os.path.join(core_path, save_filename)

    wb.save(save_address)

    rtn_filename = os.path.basename(save_address)

    return wb, rtn_filename

In [4]:
def deribit_request(method, params):
    msg = {'jsonrpc': 2.0,
           'id'     : 1,
           'method' : method,
           'params' : params
    }
    ws.send(json.dumps(msg))
    return json.loads(ws.recv())


def get_instruments(details):
    return deribit_request('public/get_instruments', details)['result']
    
def get_insts(kind, url):    
    instruments = get_instruments({'currency': 'BTC', 'kind'    : kind, 'expired' : False})    
    df = pd.DataFrame(instruments)    
    return df


def get_ticker(name):
    return deribit_request('public/ticker', {'instrument_name': name})['result']
    
def get_ticks(kind, url, instruments):
    rows = []    
    for inst in instruments:
        ticker = get_ticker(inst)
        rows.append(ticker)       
    df = pd.DataFrame(rows)    
    return df

In [5]:
def main(deribit_url):
    
    xlApp = xw.App(visible=False)

    current_date_nyc = date.today()

    wb, save_filename = file_maintenance(xlApp, current_date_nyc)
        
    try:
        dict_of_dfs = {}
        for kind in ['future', 'option']:
            
            df_insts = get_insts(kind, deribit_url)
            df_ticks = get_ticks(kind, deribit_url, df_insts['instrument_name'].tolist())
        
            df = pd.merge(df_insts, df_ticks, on='instrument_name', how='outer')
        
            for col in ['stats', 'tick_size_steps']:
                df = xlw.flattenColumn(df, col, sep='_')
    
            if kind == 'option':
                for col in ['greeks']:  # , 'tick_size_steps_0'
                    df = xlw.flattenColumn(df, col, sep='_')

            xlw.printDFToXL(save_filename, kind + 's_raw', 'a1', df)    
    
    finally:
        ws.close()
        
        wb.save()
        wb.close()
        
        xw.apps.active.quit()

if __name__ == "__main__":
    main(deribit_url)